In [ ]:
rep=20 #Used for single trial
import os
import torch
import multiprocessing
from joblib import Parallel, delayed

tkwargs = {
    "dtype": torch.double,
    "device": torch.device("cpu"),
}

In [ ]:
from botorch.models.model_list_gp_regression import ModelListGP
from botorch.models.transforms.outcome import Standardize
from gpytorch.mlls.sum_marginal_log_likelihood import SumMarginalLogLikelihood
from botorch.utils.transforms import unnormalize, normalize
from botorch.utils.sampling import draw_sobol_samples
from botorch.acquisition import AcquisitionFunction
from botorch.models import SingleTaskGP
from gpytorch.mlls import ExactMarginalLogLikelihood
from botorch import fit_gpytorch_mll
from botorch.test_functions.multi_objective import DTLZ2, GMM, ZDT2, VehicleSafety, Penicillin, CarSideImpact
NOISE_SE = torch.tensor([0.00, 0.00], **tkwargs)
#NOISE_SE = torch.tensor([0.00, 0.00], **tkwargs)
#problem = DTLZ2(num_objectives=3, dim=4, negate=True).to(**tkwargs)
#problem = CarSideImpact(negate=True).to(**tkwargs)
#problem = ZDT2(dim=6, negate=True).to(**tkwargs)
#problem = VehicleSafety(negate=True).to(**tkwargs)
problem = GMM(negate=True).to(**tkwargs)
def initialize_model(train_x, train_obj):
    # define models for objective and constraint
    #train_x = normalize(train_x, problem.bounds)
    models = []
    for i in range(train_obj.shape[-1]):
        train_y = train_obj[..., i : i + 1]
        train_yvar = torch.full_like(train_y, NOISE_SE[i] ** 2)
        models.append(
            SingleTaskGP(
                train_x, train_y, train_yvar, outcome_transform=Standardize(m=1)
            )
        )
    model = ModelListGP(*models)
    mll = SumMarginalLogLikelihood(model.likelihood, model)
    return mll, model

def generate_initial_data(n=10, seed=42):
    # generate training data
    train_x_unnormalized = draw_sobol_samples(bounds=problem.bounds, n=n, q=1, seed=seed).squeeze(1)
    train_x = normalize(train_x_unnormalized, problem.bounds)
    train_obj_true = problem(train_x_unnormalized)
    train_obj = train_obj_true + torch.randn_like(train_obj_true) * NOISE_SE
    return train_x, train_obj, train_obj_true

def plot_X_PF(X_train_np, q):
    
    # Define step numbers
    s = 60
    
    # Define the 2D range for x and y
    x = torch.linspace(problem.bounds[0][0], problem.bounds[1][0], s)  # 100 points between 0 and 1
    y = torch.linspace(problem.bounds[0][1], problem.bounds[1][1], s)
    
    # Create a 2D grid of x and y values
    X, Y = torch.meshgrid(x, y, indexing='ij')

    # Prepare the input grid for the BraninCurrin function
    input_grid = torch.stack([X, Y], dim=-1).reshape(-1, 2)  # Reshape to [N, 2] for function input

    # Compute the objective values using BraninCurrin
    Z = problem(input_grid.to(**tkwargs))
    # Plot the candidates against each other
    plt.figure(figsize=(6, 6))
    plt.contourf(X.cpu().detach().numpy(), Y.cpu().detach().numpy(), Z[:, 0].reshape(s, s).cpu().detach().numpy(), levels=50, cmap='viridis')
    #plt.scatter(X_test_all[X_pf_true_indices, 0], X_test_all[X_pf_true_indices, 1], color='r', label='Pareto Set')
    plt.scatter(X_train_np[:11, 0], X_train_np[:11, 1], color='black', marker = 'x', label='Training Data')
    plt.scatter(X_train_np[11:, 0], X_train_np[11:, 1], color='black', marker = '*', label='Training Data')
    plt.scatter(X_train_np[-q:, 0], X_train_np[-q:, 1], color='r', marker = '*', label='Training Data')
    plt.xlabel('x1')
    plt.ylabel('x2')
    plt.title('objective1 and x sampling location')
    plt.show()
    plt.figure(figsize=(6, 6))
    plt.contourf(X.cpu().detach().numpy(), Y.cpu().detach().numpy(), Z[:, 1].reshape(s, s).cpu().detach().numpy(), levels=50, cmap='viridis')
    #plt.scatter(X_test_all[X_pf_true_indices, 0], X_test_all[X_pf_true_indices, 1], color='r', label='Pareto Set')
    plt.scatter(X_train_np[:11, 0], X_train_np[:11, 1], color='black', marker = 'x', label='Training Data')
    plt.scatter(X_train_np[11:, 0], X_train_np[11:, 1], color='black', marker = '*', label='Training Data')
    plt.scatter(X_train_np[-q:, 0], X_train_np[-q:, 1], color='r', marker = '*', label='Training Data')
    plt.xlabel('x1')
    plt.ylabel('x2')
    plt.title('objective2 and x sampling location')
    plt.show()

In [ ]:
import numpy as np
import torch
import itertools
import matplotlib.pyplot as plt
from scipy.stats import mvn
from torch.distributions.normal import Normal

def build_extended_grid(Y: torch.Tensor):
    edges = []
    device = Y.device
    dtype = Y.dtype

    neg_inf = torch.tensor(-float("inf"), device=device, dtype=dtype)
    pos_inf = torch.tensor(float("inf"), device=device, dtype=dtype)

    for d in range(Y.shape[1]):
        vals = torch.unique(Y[:, d])
        vals = torch.sort(vals).values
        edges_d = torch.cat([neg_inf.view(1), vals, pos_inf.view(1)])
        edges.append(edges_d)

    return edges

def is_dominated(Y, x):
    Y_np = Y.numpy()
    dom_all = (Y_np >= x).all(axis=1)
    dom_strict = (Y_np > x).any(axis=1)
    return (dom_all & dom_strict).any()

def find_non_dominated_regions(Y, edges):
    regions = []

    for idx in itertools.product(*[range(len(e) - 1) for e in edges]):
        bounds = [(edges[k][i], edges[k][i + 1]) for k, i in enumerate(idx)]

        sample = np.array([
            lo + 1 if np.isinf(lo)
            else hi - 1 if np.isinf(hi)
            else 0.5 * (lo + hi)
            for lo, hi in bounds
        ])

        if not is_dominated(Y, sample):
            regions.append(bounds)

    return regions

def truncated_gaussian_expectation(mu, sigma, lo, hi, eps=1e-12):
    """
    Compute E[X | lo <= X <= hi] for X ~ N(mu, sigma^2)
    Fully differentiable in PyTorch.
    """
    normal = Normal(0.0, 1.0)

    alpha = (lo - mu) / sigma
    beta  = (hi - mu) / sigma

    # safe CDF
    cdf_alpha = torch.clamp(normal.cdf(alpha), min=eps, max=1-eps)
    cdf_beta  = torch.clamp(normal.cdf(beta),  min=eps, max=1-eps)

    Z = cdf_beta - cdf_alpha
    Z = torch.clamp(Z, min=eps)  # prevent division by zero

    # Use PDF directly from Normal
    pdf_alpha = torch.exp(normal.log_prob(alpha))
    pdf_beta  = torch.exp(normal.log_prob(beta))

    return mu + sigma * (pdf_alpha - pdf_beta) / Z   


def region_probability(mu, sigma, lo, hi):
    """
    Computes P(lo <= X <= hi) for independent Gaussians (Diagonal Covariance).
    Complexity: O(D) instead of O(2^D).
    
    Inputs:
        mu:    (N, D)
        sigma: (N, D) - Standard deviation (scale)
        lo:    (N, D)
        hi:    (N, D)
    """
    # 1. Standardize the bounds (Z-scores)
    # Z = (X - mu) / sigma
    z_lo = (lo - mu) / sigma
    z_hi = (hi - mu) / sigma
    
    # 2. Compute CDF for Standard Normal N(0,1)
    # torch.special.ndtr is the standard Normal CDF function
    cdf_lo = torch.special.ndtr(z_lo)
    cdf_hi = torch.special.ndtr(z_hi)
    
    # 3. Probability per dimension
    # P(l < X < h) = CDF(h) - CDF(l)
    dim_probs = cdf_hi - cdf_lo
    
    # 4. Handle Numerical Noise
    # Ensure prob is non-negative (can occur if lo approx hi)
    dim_probs = torch.clamp(dim_probs, min=0.0)
    
    # 5. Product across dimensions
    # Since dimensions are independent, total prob is product of marginals
    total_prob = dim_probs.prod(dim=-1)
    
    return total_prob    

def expected_locations_and_weights(regions, mu, sigma, clip=1e5, eps=1e-12):
    t0 = time.monotonic()

    # 1. Standardize Inputs
    if mu.dim() == 1:
        mu = mu.unsqueeze(0)
    if sigma.dim() == 1:
        sigma = sigma.unsqueeze(0)

    device = mu.device
    dtype = mu.dtype
    
    # 2. Process Regions
    if not isinstance(regions, torch.Tensor):
        regions_tensor = torch.tensor(regions, device=device, dtype=dtype)
    else:
        regions_tensor = regions

    # Unpack bounds (handling the shape we discussed: R x 2 x D)
    # If input is (R, D, 2), change indices to [:, :, 0] and [:, :, 1]
    # Assuming standard format (R, D, 2):
    lo_fixed = regions_tensor[..., 0]
    hi_fixed = regions_tensor[..., 1]

    # Handle Clipping
    lo_fixed = torch.where(torch.isinf(lo_fixed), torch.full_like(lo_fixed, -clip), lo_fixed)
    hi_fixed = torch.where(torch.isinf(hi_fixed), torch.full_like(hi_fixed, clip), hi_fixed)

    # 3. Setup Broadcasting
    K, D = mu.shape
    R = lo_fixed.shape[0]

    # Expand Mu/Sigma: (K, D) -> (K, R, D) -> Flat (K*R, D)
    mu_flat = mu.unsqueeze(1).expand(K, R, D).reshape(-1, D)
    sigma_flat = sigma.unsqueeze(1).expand(K, R, D).reshape(-1, D)

    # Expand Regions: (R, D) -> (K, R, D) -> Flat (K*R, D)
    lo_flat = lo_fixed.unsqueeze(0).expand(K, R, D).reshape(-1, D)
    hi_flat = hi_fixed.unsqueeze(0).expand(K, R, D).reshape(-1, D)

    # 4. Compute Expectations
    expected_flat = truncated_gaussian_expectation(
        mu_flat, sigma_flat, lo_flat, hi_flat
    )

    # 5. Compute Probabilities (OPTIMIZED)
    # We pass mu_flat and sigma_flat directly. No 'rv' object needed.
    probs_flat = region_probability(mu_flat, sigma_flat, lo_flat, hi_flat)

    # 6. Reshape & Normalize
    expected_locations = expected_flat.view(K, R, D)
    region_probs = probs_flat.view(K, R)

    region_probs = region_probs + eps
    region_probs = region_probs / region_probs.sum(dim=1, keepdim=True)

    t1 = time.monotonic()
    
    return expected_locations, region_probs    
    
    
def resample_from_expected(expected_pts, weights, n_samples):
    """
    expected_pts: (K, R, d)
    weights:      (K, R)      (already normalized per K)
    n_samples:    int or list of int per K

    returns:
        samples: list of arrays, samples[k] shape = (n_samples_k, d)
    """
    #expected_pts = np.asarray(expected_pts)
    #weights = np.asarray(weights)

    K, R, d = expected_pts.shape

    if not torch.is_tensor(n_samples):
        n_samples = torch.full((K,), n_samples, device=weights.device, dtype=torch.long)


    samples = []

    for k in range(K):
        idx = torch.multinomial(
            weights[k],            # (R,)
            n_samples[k],           # int
            replacement=True
        )
        
        samples.append(expected_pts[k, idx])
    samples = torch.stack(samples, dim=0) 
    return samples

def resample_from_expected_parallel(expected_pts, weights, n_samples):
    """
    Parallel resampling from expected points without python loops.

    expected_pts: (p, K, R, d)
    weights:      (p, K, R)
    n_samples:    int or tensor of shape (K,) or (p,K)

    returns:
        samples: (max_samples, p, K, d)
    """
    p, K, R, d = expected_pts.shape
    device = expected_pts.device

    # 1. Standardize n_samples to (p, K)
    if isinstance(n_samples, int):
        n_samples_tensor = torch.full((p, K), n_samples, device=device, dtype=torch.long)
        max_samples = n_samples
    else:
        if n_samples.dim() == 1: # (K,)
            n_samples_tensor = n_samples.unsqueeze(0).expand(p, K)
        else:
            n_samples_tensor = n_samples
        max_samples = n_samples_tensor.max().item()

    # 2. Reshape for Batch Processing
    # We merge 'p' and 'K' into a single batch dimension 'N'
    # weights: (p, K, R) -> (p*K, R)
    flat_weights = weights.reshape(-1, R) 
    
    # 3. Vectorized Sampling
    # torch.multinomial works on 2D input (batch, categories)
    # We sample 'max_samples' for EVERY row in the batch at once.
    # flat_idx shape: (p*K, max_samples)
    flat_idx = torch.multinomial(flat_weights, max_samples, replacement=True)

    # 4. Handle Variable Sample Counts (Masking) if n_samples varies
    # If all n_samples are the same int, we skip this for speed.
    if not isinstance(n_samples, int):
        # Create a range mask: (1, max_samples)
        range_tensor = torch.arange(max_samples, device=device).unsqueeze(0)
        # Flatten n_samples: (p*K, 1)
        flat_n_samples = n_samples_tensor.reshape(-1, 1)
        # Create mask: (p*K, max_samples)
        mask = range_tensor < flat_n_samples
        # Where mask is False (padding), we can just repeat the 0-th index 
        # or leave as is since we will likely ignore padded values downstream.
        # But to be safe, let's just keep the indices valid (they are already valid from multinomial).
        # We generally don't zero them out because gather needs valid indices.
        pass 

    # 5. Gather Samples
    # expected_pts: (p, K, R, d) -> (p*K, R, d)
    flat_pts = expected_pts.reshape(-1, R, d)

    # Expand indices for gather
    # flat_idx: (p*K, max_samples) -> (p*K, max_samples, d)
    idx_exp = flat_idx.unsqueeze(-1).expand(-1, -1, d)

    # Gather: samples_flat shape (p*K, max_samples, d)
    samples_flat = torch.gather(flat_pts, 1, idx_exp)

    # 6. Reshape back to (p, K, max_samples, d)
    samples_full = samples_flat.view(p, K, max_samples, d)

    # 7. Permute to desired output: (max_samples, p, K, d)
    samples = samples_full.permute(2, 0, 1, 3)

    return samples

def cdf_total_weak(A, B):
    """
    A: (N, D)
    B: (M, D)
    return: total number of points in B weakly dominated by any point in A
    (counts equality as domination)
    """
    # pairwise comparisons: (N, M, D)
    le = A[:, None, :] >= B[None, :, :]  # all dimensions >=

    # weak dominance: all dimensions satisfy >=
    dominates = le.all(dim=-1)  # (N, M)

    # check if any point in A weakly dominates each point in B: (M,)
    dominated_by_any = dominates.any(dim=0)

    # count total dominated points
    return dominated_by_any.sum()


A = torch.tensor([[1.0, 2.0], [3.0, 1.0]])
B = torch.tensor([[1.0, 2.0], [2.0, 2.0], [4.0, 0.0]])

print(cdf_total_weak(A, B)) 

def cdf_total_batch(A_batch, B):
    """
    A_batch: (K, N, D)  K sets of points
    B: (M, D)
    return: (K,) tensor, total number of points in B dominated by each A[k]
    """
    K, N, D = A_batch.shape
    M = B.shape[0]

    # pairwise comparisons: (K, N, M, D)
    le = A_batch[:, :, None, :] >= B[None, None, :, :]  # >= for weak dominance

    # weak dominance: all dimensions satisfy >=
    dominates = le.all(dim=-1)  # (K, N, M)

    # check if any point in each A[k] dominates each point in B: (K, M)
    dominated_by_any = dominates.any(dim=1)

    # count total dominated points for each A[k]
    return dominated_by_any.sum(dim=1)

A_batch = torch.tensor([
    [[1.0, 2.0], [3.0, 1.0]],   # A[0]
    [[2.0, 2.0], [4.0, 0.0]]    # A[1]
])
B = torch.tensor([[1.0, 2.0], [2.0, 2.0], [4.0, 0.0]])

print(cdf_total_batch(A_batch, B))


def cdf_total_batch_vectorized(f_samples, all_samples):
    """
    f_samples: (S, Q, N, D)  multiple sets of points
    all_samples: (M, D)
    
    returns: (S, Q) tensor, total number of points in all_samples dominated by each f_samples[s, q]
    """
    S, Q, N, D = f_samples.shape
    M = all_samples.shape[0]

    # expand for pairwise comparison: (S, Q, N, M, D)
    le = f_samples[:, :, :, None, :] >= all_samples[None, None, None, :, :]  # weak dominance

    # check weak dominance over all dimensions: (S, Q, N, M)
    dominates = le.all(dim=-1)

    # check if any point in each set dominates each point in all_samples: (S, Q, M)
    dominated_by_any = dominates.any(dim=2)

    # count total dominated points: (S, Q)
    dominance_count = dominated_by_any.sum(dim=-1)
    return dominance_count


In [ ]:
import itertools

def make_region_combinations(R, q, device):
    """
    Returns:
        region_idx: (R^q, q)
        Each row is a batch combination of region indices
    """
    combos = list(itertools.product(range(R), repeat=q))
    return torch.tensor(combos, device=device)

def build_all_batches(expected_pt, region_idx):
    """
    expected_pt: (P, q, R, M)
    region_idx:  (N, q)

    returns:
        Y_all: (P, N, q, M)
    """
    P, q, R, M = expected_pt.shape
    N = region_idx.shape[0]

    # For each batch position j, gather regions for all P
    Y_parts = []
    for j in range(q):
        # expected_pt[:, j, :, :] -> (P, R, M)
        # region_idx[:, j]        -> (N,)
        Yj = expected_pt[:, j, region_idx[:, j], :]   # (P, N, M)
        Y_parts.append(Yj)

    # stack over batch dimension
    Y_all = torch.stack(Y_parts, dim=2)  # (P, N, q, M)
    return Y_all


def build_all_weights(weight, region_idx):
    """
    weight:     (P, q, R)
    region_idx: (N, q)

    returns:
        W_all: (P, N)
    """
    P, q, R = weight.shape
    N = region_idx.shape[0]

    W_parts = []
    for j in range(q):
        # weight[:, j, :] -> (P, R)
        # region_idx[:, j] -> (N,)
        Wj = weight[:, j, region_idx[:, j]]   # (P, N)
        W_parts.append(Wj)

    W_all = torch.prod(torch.stack(W_parts, dim=2), dim=2)  # (P, N)
    return W_all



In [ ]:
from __future__ import annotations

import warnings
from abc import abstractmethod
from copy import deepcopy
from itertools import combinations
from typing import Any, Callable, List, Optional, Union

import torch
from botorch.acquisition.acquisition import AcquisitionFunction
from botorch.acquisition.multi_objective.objective import (
    IdentityMCMultiOutputObjective,
    MCMultiOutputObjective,
)
from botorch.acquisition.multi_objective.utils import (
    prune_inferior_points_multi_objective,
)
from botorch.exceptions.errors import UnsupportedError
from botorch.exceptions.warnings import BotorchWarning
from botorch.models.model import Model
from botorch.posteriors.posterior import Posterior
from botorch.sampling.normal import MCSampler, SobolQMCNormalSampler
from botorch.utils.multi_objective.box_decompositions.box_decomposition_list import (
    BoxDecompositionList,
)
from botorch.utils.multi_objective.box_decompositions.dominated import (
    DominatedPartitioning,
)
from botorch.utils.multi_objective.box_decompositions.non_dominated import (
    FastNondominatedPartitioning,
    NondominatedPartitioning,
)
from botorch.utils.multi_objective.box_decompositions.utils import (
    _pad_batch_pareto_frontier,
)
from botorch.utils.objective import apply_constraints_nonnegative_soft
from botorch.utils.torch import BufferDict
from botorch.utils.transforms import (
    concatenate_pending_points,
    match_batch_shape,
    t_batch_mode_transform,
)
from torch import Tensor
from torch.distributions.multivariate_normal import MultivariateNormal

class MultiObjectiveMCAcquisitionFunction(AcquisitionFunction):
    r"""Abstract base class for Multi-Objective batch acquisition functions."""

    def __init__(
        self,
        model: Model,
        previous_Y,
        sampler: Optional[MCSampler] = None,
        objective: Optional[MCMultiOutputObjective] = None,
        X_pending: Optional[Tensor] = None,
    ) -> None:
        r"""Constructor for the MCAcquisitionFunction base class.

        Args:
            model: A fitted model.
            sampler: The sampler used to draw base samples. Defaults to
                `SobolQMCNormalSampler(num_samples=128, collapse_batch_dims=True)`.
            objective: The MCMultiOutputObjective under which the samples are
                evaluated. Defaults to `IdentityMultiOutputObjective()`.
            X_pending:  A `m x d`-dim Tensor of `m` design points that have
                points that have been submitted for function evaluation
                but have not yet been evaluated.
        """
        super().__init__(model=model)
        if sampler is None:
            sampler = SobolQMCNormalSampler(num_samples=128, collapse_batch_dims=True)
        self.add_module("sampler", sampler)
        if objective is None:
            objective = IdentityMCMultiOutputObjective()
        elif not isinstance(objective, MCMultiOutputObjective):
            raise UnsupportedError(
                "Only objectives of type MCMultiOutputObjective are supported for "
                "Multi-Objective MC acquisition functions."
            )
        self.add_module("objective", objective)
        self.X_pending = None
        self.previous_Y = previous_Y
        if X_pending is not None:
            self.set_X_pending(X_pending)


    def forward(self, X: Tensor) -> Tensor:
        r"""Takes in a `batch_shape x q x d` X Tensor of t-batches with `q` `d`-dim
        design points each, and returns a Tensor with shape `batch_shape'`, where
        `batch_shape'` is the broadcasted batch shape of model and input `X`. Should
        utilize the result of `set_X_pending` as needed to account for pending function
        evaluations.
        """
        pass  # pragma: no cover




class qNRFS_EHVI(MultiObjectiveMCAcquisitionFunction):
    def __init__(
        self,
        model: Model,
        previous_Y,
        ref_point: Union[List[float], Tensor],
        partitioning: NondominatedPartitioning,
        sampler: Optional[MCSampler] = None,
        objective: Optional[MCMultiOutputObjective] = None,
        constraints: Optional[List[Callable[[Tensor], Tensor]]] = None,
        X_pending: Optional[Tensor] = None,
        eta: float = 1e-3,
    ) -> None:
        r"""q-Expected Hypervolume Improvement supporting m>=2 outcomes.

        See [Daulton2020qehvi]_ for details.

        Example:
            >>> model = SingleTaskGP(train_X, train_Y)
            >>> ref_point = [0.0, 0.0]
            >>> qEHVI = qExpectedHypervolumeImprovement(model, ref_point, partitioning)
            >>> qehvi = qEHVI(test_X)

        Args:
            model: A fitted model.
            ref_point: A list or tensor with `m` elements representing the reference
                point (in the outcome space) w.r.t. to which compute the hypervolume.
                This is a reference point for the objective values (i.e. after
                applying`objective` to the samples).
            partitioning: A `NondominatedPartitioning` module that provides the non-
                dominated front and a partitioning of the non-dominated space in hyper-
                rectangles. If constraints are present, this partitioning must only
                include feasible points.
            sampler: The sampler used to draw base samples. Defaults to
                `SobolQMCNormalSampler(num_samples=128, collapse_batch_dims=True)`.
            objective: The MCMultiOutputObjective under which the samples are evaluated.
                Defaults to `IdentityMultiOutputObjective()`.
            constraints: A list of callables, each mapping a Tensor of dimension
                `sample_shape x batch-shape x q x m` to a Tensor of dimension
                `sample_shape x batch-shape x q`, where negative values imply
                feasibility. The acqusition function will compute expected feasible
                hypervolume.
            X_pending: A `batch_shape x m x d`-dim Tensor of `m` design points that have
                points that have been submitted for function evaluation but have not yet
                been evaluated. Concatenated into `X` upon forward call. Copied and set
                to have no gradient.
            eta: The temperature parameter for the sigmoid function used for the
                differentiable approximation of the constraints.
        """
        if len(ref_point) != partitioning.num_outcomes:
            raise ValueError(
                "The length of the reference point must match the number of outcomes. "
                f"Got ref_point with {len(ref_point)} elements, but expected "
                f"{partitioning.num_outcomes}."
            )
        ref_point = torch.as_tensor(
            ref_point,
            dtype=partitioning.pareto_Y.dtype,
            device=partitioning.pareto_Y.device,
        )
        super().__init__(
            model=model, sampler=sampler, objective=objective, X_pending=X_pending, previous_Y=previous_Y,
        )
        self.constraints = constraints
        self.eta = eta
        self.register_buffer("ref_point", ref_point)
        cell_bounds = partitioning.get_hypercell_bounds()
        self.register_buffer("cell_lower_bounds", cell_bounds[0])
        self.register_buffer("cell_upper_bounds", cell_bounds[1])
        self.q = -1
        self.q_subset_indices = BufferDict()
        self.partitioning = partitioning
        n_test = 100
        X_obs=lhs(model.train_inputs[0][0].shape[-1], n_test)
        X_obs=torch.tensor(X_obs)
        
        # partition non-dominated space into disjoint rectangles
        mus = []
        covs = []
        for i in range(n_test):
            posterior = model.posterior(X_obs[i:i+1])
            mu = posterior.mean
            cov = posterior.mvn.covariance_matrix
            mus.append(mu)
            covs.append(cov)
        mus = torch.stack(mus) 
        covs = torch.stack(covs)
        
        #sigma = np.sqrt(np.array([np.diag(c) for c in cov]))
        mus = torch.tensor(mus).reshape(mus.shape[0], mus.shape[-1])
        covs = torch.tensor(covs)
        sigmas = torch.sqrt(torch.diagonal(covs, dim1=1, dim2=2))
        
        """
        Vectorized version that returns (num_samples, n_models, n_obj).
        """
        
        #torch.manual_seed(random_seed)
        n_obj = self.previous_Y.shape[1]
        n_test = mus.shape[0]
        train_pf_mask = is_non_dominated(self.previous_Y)
        y_train_pf = self.previous_Y[train_pf_mask]  # (m, n_obj)

        resolution_th = 5
        if y_train_pf.shape[0] > resolution_th:
            # 1. Separate the "Most Recent" (last one) from the "Previous" ones
            previous_pf = y_train_pf[:-1]  # All except the last
            most_recent = y_train_pf[-1:]  # The last one (keep dims for cat)

            # 2. Randomly select (resolution_th - 1) from the previous pool
            n_random = resolution_th - 1
            idx = torch.randperm(previous_pf.shape[0])[:n_random]
            selected_previous = previous_pf[idx]

            # 3. Combine them
            y_train_pf = torch.cat([selected_previous, most_recent], dim=0)

        edges = build_extended_grid(y_train_pf)
        regions = find_non_dominated_regions(y_train_pf, edges)
        expected_pts, weights = expected_locations_and_weights(
            regions, mus, sigmas
        )
        all_samples = resample_from_expected(
            expected_pts,
            weights,
            n_samples=100
        )
        all_samples = torch.tensor(all_samples).permute(1, 0, 2)
        all_nd_samples = all_samples.reshape(100 * n_test, -1)
        # compute pareto optimality
        pf_counts = []
        for i, sample in enumerate(all_samples):
            sample_pf_mask = is_non_dominated(sample) 
            sample_pf_mask = sample_pf_mask.to(torch.int)
            #sample_pf_mask += torch.ones_like(sample_pf_mask)
            pf_counts.append(sample_pf_mask)
        pf_counts = torch.stack(pf_counts, dim=0)  # (num_samples, N)
        avg_pf_counts = pf_counts.double().mean(dim=0) #+ torch.ones(n_test)/n_test
        self.X_obs = X_obs
        self.avg_pf_counts = avg_pf_counts
        self.all_nd_samples = all_nd_samples
        self.regions = regions
        self.n_obj = n_obj
        self.lengthscale = math.sqrt(self.n_obj)
        self.y_train_pf = y_train_pf
        self.ref_point = ref_point
        self.all_samples = all_samples
        self.n_test = n_test
        '''
        if ref_point.shape[0] == 2:
            plt.figure(figsize=(6, 6))
            #plt.scatter(self.all_raw_samples[:, 0], self.all_raw_samples[:, 1], alpha=0.1, c='b')
            #plt.scatter(self.all_nd_samples[:, 0], self.all_nd_samples[:, 1], alpha=0.1, c='g')
            plt.scatter(self.previous_Y[:-1, 0], self.previous_Y[:-1, 1], color='red', label='y_train', marker='x', s=50, zorder=5)
            plt.scatter(self.previous_Y[-1:, 0], self.previous_Y[-1:, 1], color='red', label='y_train', marker='*', s=50, zorder=5)
            #plt.scatter(y_train[-1:, 0], y_train[-1:, 1], color='black', label='y_train', marker='o', s=50, zorder=5)
            plt.xlabel("Objective 1")
            plt.ylabel("Objective 2")
            plt.grid(True)
            plt.tight_layout()
            plt.show()
        '''
    def _cache_q_subset_indices(self, q: int) -> None:
        r"""Cache indices corresponding to all subsets of `q`.

        This means that consecutive calls to `forward` with the same
        `q` will not recompute the indices for all (2^q - 1) subsets.

        Note: this will use more memory than regenerating the indices
        for each i and then deleting them, but it will be faster for
        repeated evaluations (e.g. during optimization).

        Args:
            q: batch size
        """
        if q != self.q:
            indices = list(range(q))
            tkwargs = {"dtype": torch.long, "device": self.cell_lower_bounds.device}
            self.q_subset_indices = BufferDict(
                {
                    f"q_choose_{i}": torch.tensor(
                        list(combinations(indices, i)), **tkwargs
                    )
                    for i in range(1, q + 1)
                }
            )
            self.q = q

    def _compute_qehvi(self, samples: Tensor, X: Optional[Tensor] = None) -> Tensor:
        r"""Compute the expected (feasible) hypervolume improvement given MC samples.

        Args:
            samples: A `n_samples x batch_shape x q x m`-dim tensor of samples.
            X: A `batch_shape x q x d`-dim tensor of inputs.

        Returns:
            A `batch_shape x (model_batch_shape)`-dim tensor of expected hypervolume
            improvement for each batch.
        """
        q = samples.shape[-2]
        # Note that the objective may subset the outcomes (e.g. this will usually happen
        # if there are constraints present).
        obj = self.objective(samples, X=X)
        if self.constraints is not None:
            feas_weights = torch.ones(
                obj.shape[:-1], device=obj.device, dtype=obj.dtype
            )
            feas_weights = apply_constraints_nonnegative_soft(
                obj=feas_weights,
                constraints=self.constraints,
                samples=samples,
                eta=self.eta,
            )
        self._cache_q_subset_indices(q=q)
        batch_shape = samples.shape[:-2]
        # this is n_samples x input_batch_shape x
        areas_per_segment = torch.zeros(
            *batch_shape,
            self.cell_lower_bounds.shape[-2],
            dtype=obj.dtype,
            device=obj.device,
        )
        cell_batch_ndim = self.cell_lower_bounds.ndim - 2
        sample_batch_view_shape = torch.Size(
            [
                batch_shape[0] if cell_batch_ndim > 0 else 1,
                *[1 for _ in range(len(batch_shape) - max(cell_batch_ndim, 1))],
                *self.cell_lower_bounds.shape[1:-2],
            ]
        )
        view_shape = (
            *sample_batch_view_shape,
            self.cell_upper_bounds.shape[-2],
            1,
            self.cell_upper_bounds.shape[-1],
        )
        for i in range(1, q + 1):
            # TODO: we could use batches to compute (q choose i) and (q choose q-i)
            # simulataneously since subsets of size i and q-i have the same number of
            # elements. This would decrease the number of iterations, but increase
            # memory usage.
            q_choose_i = self.q_subset_indices[f"q_choose_{i}"]
            # this tensor is mc_samples x batch_shape x i x q_choose_i x m
            
            obj_subsets = obj.index_select(dim=-2, index=q_choose_i.view(-1))
            obj_subsets = obj_subsets.view(
                obj.shape[:-2] + q_choose_i.shape + obj.shape[-1:]
            )
            # since all hyperrectangles share one vertex, the opposite vertex of the
            # overlap is given by the component-wise minimum.
            # take the minimum in each subset
            overlap_vertices = obj_subsets.min(dim=-2).values
            # add batch-dim to compute area for each segment (pseudo-pareto-vertex)
            # this tensor is mc_samples x batch_shape x num_cells x q_choose_i x m
            overlap_vertices = torch.min(
                overlap_vertices.unsqueeze(-3), self.cell_upper_bounds.view(view_shape)
            )
            # substract cell lower bounds, clamp min at zero
            lengths_i = (
                overlap_vertices - self.cell_lower_bounds.view(view_shape)
            ).clamp_min(0.0)
            # take product over hyperrectangle side lengths to compute area
            # sum over all subsets of size i
            areas_i = lengths_i.prod(dim=-1)
            # if constraints are present, apply a differentiable approximation of
            # the indicator function
            if self.constraints is not None:
                feas_subsets = feas_weights.index_select(
                    dim=-1, index=q_choose_i.view(-1)
                ).view(feas_weights.shape[:-1] + q_choose_i.shape)
                areas_i = areas_i * feas_subsets.unsqueeze(-3).prod(dim=-1)
            areas_i = areas_i.sum(dim=-1)
            # Using the inclusion-exclusion principle, set the sign to be positive
            # for subsets of odd sizes and negative for subsets of even size
            areas_per_segment += (-1) ** (i + 1) * areas_i
        # sum over segments and average over MC samples
        return areas_per_segment.sum(dim=-1).mean(dim=0)
        
    def _compute_pf_probability(self, X: Tensor) -> Tensor:
        B, S, d = X.shape
        X_flat = X.reshape(B * S, d)
        d2 = torch.cdist(X_flat, self.X_obs).pow(2)
        logW = -0.5 * d2 / self.lengthscale**2
        logW = logW - logW.max(dim=1, keepdim=True).values
        W = torch.exp(logW)
        W = W / W.sum(dim=1, keepdim=True)
        y_flat = (W @ self.avg_pf_counts.unsqueeze(-1)).squeeze(-1)
        y_gmm = y_flat.view(B, S)
        y_gmm_prior = y_gmm + 0.01
        y_out = y_gmm_prior.prod(dim=1)
        #y_out = y_gmm.sum(dim=1)
        return y_out


    def forward(self, X: Tensor) -> Tensor:
        """
        X: (batch_shape, 1, d)
        """

        #y_train_index = is_non_dominated(y_trains)
        #y_train_pf = all_raw_samples[y_train_index]
        #print(all_raw_samples.shape)
        #all_nd_samples_y_train_pf = torch.cat((all_nd_samples, y_train_pf))
        #print(all_raw_samples_y_train.shape)


        posterior = self.model.posterior(X)
        mu = posterior.mean.reshape(-1, self.n_obj)
        cov = []
        q = X.shape[1]
        for i in range(q):
            posterior_s = self.model.posterior(X[:,i:i+1,:])
            cov_s = posterior_s.mvn.covariance_matrix
            cov.append(cov_s)
        cov = torch.stack(cov, dim=1).reshape(-1, self.n_obj, self.n_obj)
        sigma = torch.sqrt(torch.diagonal(cov, dim1=1, dim2=2))
        expected_pt, weight = expected_locations_and_weights(
            self.regions, mu, sigma
        )
        region_num = expected_pt.shape[-2]
        expected_pt = expected_pt.reshape(-1, q, region_num, self.n_obj)
        weight = weight.reshape(-1, q, region_num)
        mcn = 100
        f_samples = resample_from_expected_parallel(
            expected_pt,
            weight,
            n_samples=mcn
        )
        
        #dominance_counts = cdf_total_batch_vectorized(f_samples, self.all_samples.reshape(-1, self.n_obj))
        #ehvi = dominance_counts.float().mean(dim=0) / (self.n_test * mcn)
        f_samples = f_samples.view(mcn, X.shape[0], q, self.n_obj)
        ehvi = self._compute_qehvi(samples=f_samples)
        '''
        dominance_counts = torch.stack(dominance_counts, dim=0)  # (num_samples, N)
        avg_dominance_counts = dominance_counts.float().mean(dim=0)
        #print(avg_dominance_counts)
        '''    

        '''
        posterior = self.model.posterior(X)
        f_samples = self.sampler(posterior)
        ehvi = self._compute_qehvi(samples=f_samples)
        '''
        
        avg_pf_p = self._compute_pf_probability(X)
        return avg_pf_p * ehvi

In [ ]:
from botorch.models.model import Model
from typing import Any, Callable, Optional
from botorch.optim.optimize import optimize_acqf, optimize_acqf_list
from botorch.acquisition.objective import GenericMCObjective
from botorch.utils.multi_objective.scalarization import get_chebyshev_scalarization
from botorch.utils.multi_objective.box_decompositions.non_dominated import (
    FastNondominatedPartitioning,
)
from botorch.utils.multi_objective.box_decompositions.dominated import (
    DominatedPartitioning,
)
#from EHVI_batch_filling import qExpectedHypervolumeImprovementFilling
from botorch.utils.sampling import sample_simplex
from botorch.acquisition.multi_objective.utils import (
    sample_optimal_points,
    random_search_optimizer,
    compute_sample_box_decomposition
)
from botorch.sampling.normal import SobolQMCNormalSampler
from torch import Tensor
from botorch.utils.multi_objective.pareto import is_non_dominated


NUM_RESTARTS = 5 
RAW_SAMPLES = 64 


def optimize_qehvi_and_get_observation(model, train_x, train_obj, sampler, q):
    """Optimizes the qEHVI acquisition function, and returns a new candidate and observation."""
    n_obj = train_obj.shape[-1]
            
    '''
    '''
    with torch.no_grad():
        pred = model.posterior(train_x).mean
        
    partitioning = FastNondominatedPartitioning(
        ref_point=torch.tensor(problem.ref_point).reshape(-1).to(**tkwargs),
        Y=pred,
    )
    
    acq_func = qNRFS_EHVI(
        model=model,
        previous_Y = train_obj,
        ref_point=torch.tensor(problem.ref_point).reshape(-1).to(**tkwargs),
        partitioning=partitioning,
        sampler=sampler,
    )

    candidates, _ = optimize_acqf(
        acq_function=acq_func,
        bounds=problem.bounds,
        q=q,
        num_restarts=NUM_RESTARTS,
        raw_samples=RAW_SAMPLES, 
        options={"batch_limit": 5, "maxiter": 200},
        sequential=False,
        # set equality constraints to make sure sum of composition is 1
        #equality_constraints=[(indices, coefficients, rhs)]
        #inequality_constraints=... if needed
    )
    # Compute pairwise distances: result shape (M, N)
    #distances = np.linalg.norm(X_test[None, :, :] - candidates[:, None, :], axis=2)
    
    '''
    distances = np.linalg.norm(X_test_all[None, :, :] - candidates[:, None, :], axis=2)
    nearest_indices = np.argmin(distances, axis=1)
    
    new_x = X_test_all[nearest_indices].reshape(q, N_dim)
    new_x_unnormalized = X_test_all_unnormalized[nearest_indices]
    new_obj_true = problem(new_x_unnormalized)    
    new_obj = new_obj_true
    '''
    
    new_x = candidates
    new_x_unnormalized = unnormalize(new_x, problem.bounds) 
    new_obj_true = problem(new_x_unnormalized)
    new_obj = new_obj_true
    
    new_train_obj = torch.cat((train_obj, new_obj))
    plt.scatter(train_obj[:, 0], train_obj[:, 1], color='red', label='y_train', marker='x', s=50, zorder=5)
    plt.xlabel("Objective 1")
    plt.ylabel("Objective 2")
    plt.grid(True)
    plt.tight_layout()
    plt.show()    
    return new_x, new_obj, new_obj_true

In [ ]:
verbose = True

# Define a function d(X, x*) to calculate minimum distance between set X and X*
def distance_XX(X, X_star):
    d_sum = 0
    for x_star in X_star:
        d_list = torch.norm(X - x_star, dim=1)
        d_sum += torch.min(d_list)
    return d_sum / len(X_star) #/ volume

In [ ]:
### import torch
import gpytorch
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import norm
from pyDOE import *
from copy import deepcopy
import os
import shutil
from multiprocessing import Pool
import multiprocessing
from joblib import Parallel, delayed
import random
warnings.filterwarnings("ignore")

itr=1000
N_dim=2
N_test=1500
# Should be 100, 10 is for testing
#N_alt=100
N_alt=10
N_samp=1
N_obj=2
MC_SAMPLES = 128
BATCH_SIZE = 1
#chosen_acq='EHVI'
#chosen_acq='EHVI_Botorch'
#chosen_acq='PES_Botorch'
chosen_acq='NRFS_Botorch'
#chosen_acq='TPE_Optuna'
#chosen_acq='RS_Botorch'


hv_total=[]
edmin_total = []
exploration_total = []
exploitation_total = []

for j in range(15,16):
    train_x_nrfs, train_y_nrfs, _ = generate_initial_data(seed=j)
    data_x_nrfs=train_x_nrfs.detach().numpy()
    data_y_nrfs=train_y_nrfs.detach().numpy()
    train_x_nrfs=torch.tensor(train_x_nrfs).to(**tkwargs)
    train_y_nrfs=torch.tensor(train_y_nrfs).to(**tkwargs)
    mll_nrfs, model_nrfs = initialize_model(train_x_nrfs, train_y_nrfs)
    
    
    X_test_all=lhs(N_dim,N_test)
    X_test_all=torch.tensor(X_test_all)
    X_test_all_unnormalized=unnormalize(X_test_all, problem.bounds)
    Y_test_all = problem(X_test_all_unnormalized)
    X_test_all = torch.tensor(X_test_all).to(**tkwargs)
    
    
    # Find PF ground truth and calculate HV
    pareto_mask_test_all = is_non_dominated(Y_test_all)
    Y_pf = Y_test_all[pareto_mask_test_all]  
    bd_test_all = DominatedPartitioning(ref_point=problem.ref_point, Y=Y_pf)
    volume_test_all = bd_test_all.compute_hypervolume().item()
    
    # Initialize edmin
    X_pf = torch.tensor(X_test_all[pareto_mask_test_all]).to(**tkwargs) 
    edmin = distance_XX(train_x_nrfs, X_pf).reshape(1,1)
    
    # Initialize hv
    pareto_mask_train = is_non_dominated(train_y_nrfs)
    Y_pf_train = train_y_nrfs[pareto_mask_train] 
    bd_train = DominatedPartitioning(ref_point=problem.ref_point, Y=Y_pf_train)
    hv_truth = np.array(bd_train.compute_hypervolume().item()).reshape(1,1)   
    
    iteration=0
    if chosen_acq == 'NRFS_Botorch':
    
        while iteration<itr:
            plot_X_PF(train_x_nrfs, BATCH_SIZE)
            iteration += 1
            t0 = time.monotonic()
            
            # Candidates initialization
            #X_test=lhs(N_dim,N_alt)
            #X_test=torch.tensor(X_test)
            #X_test_unnormalized=unnormalize(X_test, problem.bounds)
            # Fit the models
            fit_gpytorch_mll(mll_nrfs)
            
            # Define the qEHVI acquisition module using a QMC sampler
            nrfs_sampler = SobolQMCNormalSampler(sample_shape=torch.Size([MC_SAMPLES]))

            # Optimize acquisition function and get new observations
            new_x_nrfs, new_y_nrfs, new_y_true_nrfs = optimize_qehvi_and_get_observation(
                model_nrfs, train_x_nrfs, train_y_nrfs, nrfs_sampler, BATCH_SIZE
            )

            # Update training points
            train_x_nrfs = torch.cat([train_x_nrfs, torch.tensor(new_x_nrfs).to(**tkwargs).reshape(-1, N_dim)])
            train_y_nrfs = torch.cat([train_y_nrfs, torch.tensor(new_y_nrfs).to(**tkwargs).reshape(-1, N_obj)])
            data_x_nrfs=train_x_nrfs.cpu().detach().numpy()
            data_y_nrfs=train_y_nrfs.cpu().detach().numpy()
            
            # Compute hypervolume
            pareto_mask_train = is_non_dominated(train_y_nrfs)
            Y_pf_train = train_y_nrfs[pareto_mask_train] 
            bd_train = DominatedPartitioning(ref_point=problem.ref_point, Y=Y_pf_train)
            hv_t = np.array(bd_train.compute_hypervolume().item())    
            hv_truth=np.concatenate((hv_truth,hv_t.reshape(1,1)))

            # Compute edmin
            ed_t = distance_XX(train_x_nrfs, X_pf).reshape(1,1)
            edmin = torch.cat((edmin, ed_t.reshape(1,1)))
            
            # Reinitialize the models for next iteration
            mll_nrfs, model_nrfs = initialize_model(train_x_nrfs, train_y_nrfs)
            t1 = time.monotonic()

            if verbose:
                print("Iteration:", iteration)
                print('new candidats:', new_x_nrfs) 
                print('new obj:', new_y_nrfs)
                print("Hypervolume (qPES):", hv_truth[-1])
                print("Time:", t1 - t0)
     
        pd.DataFrame(Y_pf_train).to_csv("y_pareto_truth"+str(j)+".csv", header=None, index=None)
        pd.DataFrame(data_x_nrfs).to_csv("data_x"+str(j)+".csv", header=None, index=None)
        pd.DataFrame(data_y_nrfs).to_csv("data_y"+str(j)+".csv", header=None, index=None)
        pd.DataFrame(hv_truth).to_csv("hv_truth"+str(j)+".csv", header=None, index=None)

        
    hv_total.append(np.ravel(hv_truth))
    pd.DataFrame(hv_total).to_csv("hv_truth_total"+str(rep)+".csv", header=None, index=None)

    edmin_total.append(np.ravel(edmin.cpu().detach().numpy()))
    pd.DataFrame(edmin_total).to_csv("edmin_total"+str(rep)+".csv", header=None, index=None)